# Chapter 5: AI-Assisted Requirements & Planning

This notebook provides a runnable, **exhaustive companion** to Chapter 5 of the SDLC book. It follows the **GlobalBank Corporate Onboarding** case study end-to-end.

We use the standard `openai` Python SDK. By avoiding wrapper libraries like LangChain, you can see exactly how the bare-metal prompts from the book are structured and sent to the LLM.

### Prerequisites
Ensure you have the `openai` package installed, and an OpenAI API key ready.

In [ ]:
!pip install -q openai

import os
import json
import getpass
from openai import OpenAI

# Set your OpenAI API Key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

def generate_response(system_prompt, user_prompt):
    """Helper function to call the OpenAI API"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.1,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

## 1. AI-Assisted Stakeholder Discovery (§5.1)
Before eliciting requirements, we need to know who to talk to. We use AI to analyze the project description and identify stakeholders across the org chart, including hidden regulatory or downstream technical dependencies.

In [ ]:
project_description = """
GlobalBank Onboarding is a new digital platform for onboarding corporate clients. 
It needs to automate entity resolution for complex corporate structures, perform UK FCA 
KYC/AML screening in real-time by integrating with World-Check, generate risk decisioning scores, 
and automatically provision accounts in the legacy core banking system.
"""

system_prompt = """
You are a senior business analyst. Given a project description,
identify 6 distinct stakeholders who should be consulted during requirements elicitation.
Think specifically about UK banking compliance, operational risk, and technical integration.

Format as a Markdown table with columns:
| Role/Title | Department | Why Include Them (Impact/Influence) | Priority (Primary/Secondary) |
"""

response = generate_response(system_prompt, project_description)
print(response)

## 2. Requirements Elicitation & Extraction using EARS (§5.1)
We extract structured requirements from a raw meeting transcript. We specifically force the AI to use the EARS (Easy Approach to Requirements Syntax) pattern to ensure high-quality statements instead of vague notes.

In [ ]:
# Load the mock kickoff meeting transcript
with open("data/stakeholder_transcript.txt", "r") as f:
    transcript = f.read()

system_prompt = """
Analyze the stakeholder meeting transcript and extract a comprehensive requirements backlog. 
Format EVERY requirement using one of the EARS patterns:
- Ubiquitous: The <system> shall <action>
- Event-driven: When <trigger>, the <system> shall <action>
- Unwanted: If <condition>, then the <system> shall <action>

Format as a Markdown table with columns:
| ID | EARS Requirement | Type (Func/Non-Func/Compliance) | Source Speaker |
"""

extracted_reqs = generate_response(system_prompt, f"Transcript:\n{transcript}")
print(extracted_reqs)

## 3. User Story Generation (CPFC Pattern) (§5.2)
We take a single extracted requirement and generate a fully-fleshed out, sprint-ready user story using the **Context-Persona-Feature-Criteria (CPFC)** pattern discussed in the book.

In [ ]:
system_prompt = """
Convert this raw requirement into a rigorous agile User Story.

Follow this exact Context-Persona-Feature-Criteria (CPFC) format:
### Story: [Title]
**As a** [persona],
**I want** [feature],
**So that** [benefit associated with UK FCA compliance]

**Acceptance Criteria:**
(Provide at least 4 criteria using Given/When/Then syntax)

**Edge Cases:**
(List 3 edge cases, e.g. API failures, fuzzy name matches)

**Non-Functional Requirements:**
(List performance/security constraints)
"""

user_prompt = "Requirement: The system shall screen the company name and Ultimate Beneficial Owners (UBOs) against the latest OFAC and UN sanctions lists automatically in real-time."

story = generate_response(system_prompt, user_prompt)
print(story)

## 4. Specification Analysis & Gap Detection (§5.3)
AI is excellent at catching contradictions, ambiguities, and missing logic within a draft specification. We feed it a flawed set of rules to see what it catches.

In [ ]:
draft_specs = """
FR-001: The system shall automatically screen all uploaded IDs against the World-Check database.
FR-002: To maximize onboarding speed, the system shall complete the entire onboarding workflow (including all database checks) in under 10 seconds.
FR-003: If an applicant's ID matches a record on the Interpol watch list, the account must be locked and local authorities notified immediately.
FR-004: The system must strictly comply with GDPR, and delete all applicant PII immediately if they abandon the onboarding form halfway through.
"""

system_prompt = """
Act as a critical Senior Business Analyst. Review the provided draft requirements 
and run a Specification Quality Analysis. Output a categorized report identifying:

1. Contradictions (e.g. speed SLA vs external API latency constraints, Data deletion laws vs Anti-Money Laundering data retention laws)
2. Ambiguities (vague terms lacking definitions)
3. Missing Exceptions (Unhappy paths not covered)
"""

analysis = generate_response(system_prompt, f"Requirements:\n{draft_specs}")
print(analysis)

## 5. Traceability Matrix Generation (§5.3)
We provide the AI with a list of requirements, architecture components, and test cases, and ask it to link them together to generate an ISO/IEEE compliant Traceability Matrix. This highlights gaps in test coverage.

In [ ]:
with open("data/architecture_metadata.json", "r") as f:
    arch_data = json.load(f)
    
system_prompt = """
You are a requirements traceability analyst. Link the provided Requirements to their most likely 
Design Components and Test Cases. 

Generate a Markdown Traceability Matrix with columns: [Req ID, Requirement, Design Component, Test Cases, Coverage Status].
Then, clearly list any Coverage Gaps.
"""

user_prompt = f"""
REQUIREMENTS:
- REQ-1: Auto-screen entities against OFAC sanction lists
- REQ-2: Flag fuzzy name matches over 80% confidence
- REQ-3: Log all compliance checks immutably for FCA audits
- REQ-4: Provide a dashboard for Relationship Managers to check status

COMPONENTS:
{json.dumps(arch_data['components'], indent=2)}

TEST CASES:
{json.dumps(arch_data['test_cases'], indent=2)}
"""

matrix = generate_response(system_prompt, user_prompt)
print(matrix)

## 6. AI-Assisted Estimation via Historical Analogy (§5.4)
Software estimation is notoriously inaccurate. We pass a new User Story and a database of historical, completed Jira stories to the AI so it can suggest a Story Point estimate based on actual historical data.

In [ ]:
with open("data/historical_stories.json", "r") as f:
    historical_data = f.read()

new_story = """
Integrate the 'ComplyAdvantage' API for real-time adverse media and PEP (Politically Exposed Person) screening. 
We need to send the entity name and Date of Birth via their REST API, parse the JSON response for risk categorization, 
and handle fuzzy matching scores. Latency must remain under 800ms to avoid timing out the onboarding UI.
"""

system_prompt = """
Act as an Agile Scrum Master. We need to estimate a new user story.
Review the historical completed stories and their actual story points provided by the user.

Provide:
1. Which historical story is the closest analogy and why.
2. A suggested Story Point estimate (Fibonacci: 1, 2, 3, 5, 8, 13) for the New Story.
3. Key complexity drivers or risks for the New Story.
"""

estimate = generate_response(system_prompt, f"Historical Data:\n{historical_data}\n\nNew Story to Estimate:\n{new_story}")
print(estimate)

## 7. WSJF Prioritization and Epic Decomposition (§5.5)
We use AI to break down a large Epic into vertical slices, and then apply SAFe's **Weighted Shortest Job First (WSJF)** formula to automatically rank the stories by priority.

In [ ]:
with open("data/onboarding_epic.md", "r") as f:
    epic_description = f.read()

system_prompt = """
Decompose the provided Epic into exactly 5 vertical-slice User Stories.
Then, for each story, estimate parameters to calculate its WSJF (Weighted Shortest Job First) score.

Score the following out of 10:
- Business/User Value (BV)
- Time Criticality (TC) (Regulatory deadlines score highest)
- Risk Reduction/Opportunity Enablement (RR)
- Job Size / Effort (JS) (1 is easiest, 10 is hardest)

WSJF Equation = (BV + TC + RR) / JS

Output a Markdown table ordered from HIGHEST WSJF score to lowest. 
Include columns for Title, Description, BV, TC, RR, Cost of Delay (BV+TC+RR), Job Size, and Final WSJF Score.
"""

backlog = generate_response(system_prompt, f"Epic:\n{epic_description}")
print(backlog)